# New labels for moved plans: the test on LIBERO-PRO Object
**VLA direction** · LIBERO-PRO, Object suite · released starVLA checkpoint, VLM frozen · 16 Nov 2026 · two ten-task action heads that differ only in their labels

- **The question:** the diagnosis of 15 Nov traced three failing tasks of the task perturbation (ketchup, milk, orange juice) to the training labels for "another object named": they made a detour when the object asked for was close to the gripper's start, and copied the final approach of the scene's original target. With labels that scale the path about the start and take the final approach from the object asked for, do these tasks recover, and do the other tasks hold?
- **Real:** both action heads, trained the same way on all ten LIBERO-Object tasks except for the labels: `s9_align_release10` (labels of 13 Nov: align first) and `s10_scale_own10` (labels of 15 Nov: `--scale-from-start --own-approach`); the plan head that scores image cells (`g1`) and its plans, kept per phase; all rollouts.
- **ORACLE:** the plan of the columns marked "oracle"; the grasp points the plans are compared with.
- **Protocol:** both perturbations, 10 tasks × 20 start states, seed 7 (the protocol of the 20-start runs, not the papers' 50 states and three seeds).

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "affordance_vla").is_dir())
sys.path.insert(0, str(ROOT))
from affordance_vla.plan.plan_input import offsets_from_json
from report import libero_pro_helpers as L
from report import plan_predictor_helpers as P
from report import task_perturbation_helpers as T

pd.set_option("display.max_colwidth", 60)
PRO = ROOT / "runs" / "libero_pro"
offsets = offsets_from_json((ROOT / "runs" / "plan_following" / "dataset10" / "offsets.json").read_text())
RUNS = {"labels of 13 Nov + oracle plan": "oracle_plan_s9",
        "labels of 15 Nov + oracle plan": "oracle_plan_s10",
        "labels of 13 Nov + map plans, kept": "predicted_g1_hold_s9",
        "labels of 15 Nov + map plans, kept": "predicted_g1_hold_s10"}
runs = {name: {suite: L.load_suite(PRO / run, suite) for suite in L.SUITES.values()} for name, run in RUNS.items()}

## 1 · Solved rollouts
**This step:** rollouts in which the benchmark's goal check held, per perturbation, for the two heads with the oracle plan and with kept map plans.

**Status:** real rollouts; two columns get the ORACLE plan.

In [2]:
display(L.score_table(runs).T)

,position: the target swaps places with another object,task: the instruction and the goal name another object
labels of 13 Nov + oracle plan,185 / 200 = 0.93,152 / 200 = 0.76
labels of 15 Nov + oracle plan,186 / 200 = 0.93,166 / 200 = 0.83
"labels of 13 Nov + map plans, kept",133 / 200 = 0.67,136 / 200 = 0.68
"labels of 15 Nov + map plans, kept",129 / 200 = 0.65,151 / 200 = 0.76


## 2 · Per task
**This step:** the same counts per task. The three tasks of the diagnosis are 3 (ketchup), 4 (milk) and 6 (orange juice) of the task perturbation.

**Status:** real.

In [3]:
for label, suite in L.SUITES.items():
    print(label)
    display(L.per_task_table(runs, suite))

position: the target swaps places with another object


,instruction given,labels of 13 Nov + oracle plan,labels of 15 Nov + oracle plan,"labels of 13 Nov + map plans, kept","labels of 15 Nov + map plans, kept"
task,,,,,
0,pick up the alphabet soup and place it in the basket,18 / 20,19 / 20,17 / 20,18 / 20
1,pick up the cream cheese and place it in the basket,20 / 20,20 / 20,19 / 20,18 / 20
2,pick up the salad dressing and place it in the basket,15 / 20,16 / 20,15 / 20,12 / 20
3,pick up the bbq sauce and place it in the basket,20 / 20,19 / 20,10 / 20,11 / 20
4,pick up the ketchup and place it in the basket,14 / 20,15 / 20,9 / 20,14 / 20
5,pick up the tomato sauce and place it in the basket,19 / 20,20 / 20,13 / 20,2 / 20
6,pick up the butter and place it in the basket,20 / 20,20 / 20,12 / 20,14 / 20
7,pick up the milk and place it in the basket,20 / 20,20 / 20,11 / 20,12 / 20
8,pick up the chocolate pudding and place it in the basket,20 / 20,19 / 20,18 / 20,19 / 20


task: the instruction and the goal name another object


,instruction given,labels of 13 Nov + oracle plan,labels of 15 Nov + oracle plan,"labels of 13 Nov + map plans, kept","labels of 15 Nov + map plans, kept"
task,,,,,
0,pick up the cream cheese and place it in the basket,20 / 20,20 / 20,20 / 20,19 / 20
1,pick up the alphabet soup and place it in the basket,19 / 20,18 / 20,20 / 20,20 / 20
2,pick up the tomato sauce and place it in the basket,18 / 20,20 / 20,14 / 20,16 / 20
3,pick up the ketchup and place it in the basket,5 / 20,1 / 20,0 / 20,0 / 20
4,pick up the milk and place it in the basket,3 / 20,10 / 20,4 / 20,4 / 20
5,pick up the bbq sauce and place it in the basket,19 / 20,19 / 20,16 / 20,19 / 20
6,pick up the orange juice and place it in the basket,8 / 20,18 / 20,7 / 20,16 / 20
7,pick up the butter and place it in the basket,20 / 20,20 / 20,20 / 20,20 / 20
8,pick up the salad dressing and place it in the basket,20 / 20,20 / 20,16 / 20,19 / 20


## 3 · How the rollouts went
**This step:** the measures of the diagnosis for each task of the task perturbation and each head: when the gripper first reaches the grasp point, how often it never does or runs to the step limit, and how long it stalls far from the object; then how far the rollouts got (held, knocked over, let go outside the basket).

**Status:** real; the reach is measured against the ORACLE grasp point.

In [4]:
scene_targets = {L.task_of(log): log.target for log in runs["labels of 13 Nov + oracle plan"]["libero_object_swap"]}
for name in ("labels of 13 Nov + oracle plan", "labels of 15 Nov + oracle plan"):
    print(name)
    rows = T.task_rows(L.episode_paths(PRO / RUNS[name], "libero_object_task"), scene_targets, offsets)
    display(T.task_table(rows).drop(columns=["gripper start to grasp point (cm)",
                                             "original target to the object asked for (cm)"]))
display(pd.DataFrame({f"{name}: {label.split(':')[0]}": L.stage_counts(PRO / run, suite)
                      for name, run in RUNS.items() for label, suite in L.SUITES.items()}))

labels of 13 Nov + oracle plan


,asked for,scene of LIBERO-Object task,solved,step of first reach (median),never reached,ran to the step limit,steps stalled far away (median)
task,,,,,,,
0,cream cheese,alphabet soup,20 / 20,42.0,0,0,2.0
1,alphabet soup,cream cheese,19 / 20,52.5,0,1,3.0
2,tomato sauce,salad dressing,18 / 20,41.0,4,2,4.5
3,ketchup,bbq sauce,5 / 20,86.5,0,15,12.0
4,milk,ketchup,3 / 20,46.0,15,17,25.0
5,bbq sauce,tomato sauce,19 / 20,72.0,0,1,11.0
6,orange juice,butter,8 / 20,121.5,12,12,32.0
7,butter,milk,20 / 20,45.0,0,0,1.0
8,salad dressing,chocolate pudding,20 / 20,67.0,0,0,1.0


labels of 15 Nov + oracle plan


,asked for,scene of LIBERO-Object task,solved,step of first reach (median),never reached,ran to the step limit,steps stalled far away (median)
task,,,,,,,
0,cream cheese,alphabet soup,20 / 20,42.0,0,0,5.0
1,alphabet soup,cream cheese,18 / 20,54.0,0,2,4.0
2,tomato sauce,salad dressing,20 / 20,39.0,0,0,3.0
3,ketchup,bbq sauce,1 / 20,60.5,2,19,11.0
4,milk,ketchup,10 / 20,88.0,4,10,11.0
5,bbq sauce,tomato sauce,19 / 20,71.0,0,1,9.5
6,orange juice,butter,18 / 20,103.0,0,3,24.5
7,butter,milk,20 / 20,51.0,0,0,1.0
8,salad dressing,chocolate pudding,20 / 20,60.0,0,0,2.0


,labels of 13 Nov + oracle plan: position,labels of 13 Nov + oracle plan: task,labels of 15 Nov + oracle plan: position,labels of 15 Nov + oracle plan: task,"labels of 13 Nov + map plans, kept: position","labels of 13 Nov + map plans, kept: task","labels of 15 Nov + map plans, kept: position","labels of 15 Nov + map plans, kept: task"
rollouts,200,200,200,200,200,200,200,200
gripper came within 2 cm of the planned grasp point,200,169,199,193,190,169,187,183
object knocked over before it was held,6,11,8,13,12,6,11,12
object held,196,161,196,188,157,155,153,168
object in the basket at the end,185,152,186,166,133,136,129,151
"held, then let go outside the basket",8,6,6,11,7,5,9,8
"held, still in the gripper at the end",3,3,4,11,17,14,15,11


## 4 · Where the map plans pointed
**This step:** for the two heads with kept map plans, the first plan of each rollout against the ORACLE grasp point of the object asked for. With kept plans the first plan is the same for both heads (same start images, same plan head), so a difference in what follows comes from the action head.

**Status:** real plans; grasp points ORACLE.

In [5]:
for label, suite in L.SUITES.items():
    print(label)
    audits = {name: [P.grasp_plans(log, offsets) for log in runs[name][suite]]
              for name in RUNS if "map" in name}
    display(P.per_task_pointing(audits))

position: the target swaps places with another object


,asked for,"labels of 13 Nov + map plans, kept","labels of 15 Nov + map plans, kept"
task,,,
0,alphabet soup,plan right 20 · solved 17 / 20,plan right 20 · solved 18 / 20
1,cream cheese,plan right 20 · solved 19 / 20,plan right 20 · solved 18 / 20
2,salad dressing,plan right 20 · solved 15 / 20,plan right 20 · solved 12 / 20
3,bbq sauce,plan right 16 · solved 10 / 20,plan right 16 · solved 11 / 20
4,ketchup,plan right 20 · solved 9 / 20,plan right 20 · solved 14 / 20
5,tomato sauce,plan right 20 · solved 13 / 20,plan right 20 · solved 2 / 20
6,butter,plan right 14 · solved 12 / 20,plan right 14 · solved 14 / 20
7,milk,plan right 14 · solved 11 / 20,plan right 14 · solved 12 / 20
8,chocolate pudding,plan right 20 · solved 18 / 20,plan right 20 · solved 19 / 20


task: the instruction and the goal name another object


,asked for,"labels of 13 Nov + map plans, kept","labels of 15 Nov + map plans, kept"
task,,,
0,cream cheese,plan right 20 · solved 20 / 20,plan right 20 · solved 19 / 20
1,alphabet soup,plan right 20 · solved 20 / 20,plan right 20 · solved 20 / 20
2,tomato sauce,plan right 20 · solved 14 / 20,plan right 20 · solved 16 / 20
3,ketchup,plan right 20 · solved 0 / 20,plan right 20 · solved 0 / 20
4,milk,plan right 20 · solved 4 / 20,plan right 20 · solved 4 / 20
5,bbq sauce,plan right 20 · solved 16 / 20,plan right 20 · solved 19 / 20
6,orange juice,plan right 20 · solved 7 / 20,plan right 20 · solved 16 / 20
7,butter,plan right 20 · solved 20 / 20,plan right 20 · solved 20 / 20
8,salad dressing,plan right 20 · solved 16 / 20,plan right 20 · solved 19 / 20


## What to take from this

**What this test shows**
- **The detours are gone, and the task perturbation gains the most.** With the oracle plan the head with the new labels solves 166 of 200 under the task perturbation (152 with the labels of 13 Nov), with kept map plans 151 (136). The orange juice, whose labels made the largest detour, rises from 8 to 18 of 20 with the oracle plan and from 7 to 16 with map plans. With the oracle plan the gripper now reaches it in all 20 rollouts (8 never did before) and runs out of time in 3 instead of 12, though it still pauses on the way (a median of 25 steps nearly still, against 32).
- **The milk rises with the oracle plan (3 to 10 of 20), not with kept map plans (4 and 4).** The map plan for the milk is as good as the oracle plan (0.3 cm off, the same approach direction), so the plan is not the reason. The difference is what happens after the milk is bumped: the oracle plan follows the carton, a kept plan stays where the carton stood. With map plans the milk is knocked over in 10 of 20 rollouts.
- **The ketchup is reached sooner but still not delivered** (5 to 1 of 20 with the oracle plan). The gripper now gets to it after a median of 60 steps instead of 86 and holds it more often (12 against 10), but most of those rollouts end with the ketchup outside the basket or knocked over. Its failure has moved from the approach to the grasp and the carry.
- **The new head follows the plan more exactly, which costs where the plan is off.** Under the position perturbation the oracle-plan column is unchanged (186 against 185), but with kept map plans one task collapses: the tomato sauce, from 13 to 2 of 20 (129 against 133 in all). Its map plan is about 3 cm off; the head of 13 Nov still found the can, the new head goes to the planned point and closes on nothing.

**What it does not show**
- **That the differences are more than noise, task by task.** Each task has 20 rollouts from one seed; changes of a few rollouts per task are within noise. The totals of the task perturbation (+14 and +15 of 200) are one and a half to two standard errors of a difference; the orange juice (8 to 18, 7 to 16) and the tomato sauce (13 to 2) are well outside.
- **Which of the two changes does what.** Scaling the path and taking the object's own final approach were tested together.

**Options for the project lead**
1. Make the labels of 15 Nov the default: the gain is on the perturbation where another object is asked for, which is where a plan from the image matters most; the loss is one task where the plan head is 3 cm off.
2. Teach the head to correct small plan errors toward the object it sees: for samples that name another object, put 2 to 3 cm of noise on the plan input while the label still goes to the object's real grasp point (now 1 cm).
3. Renew a kept plan during the approach (the milk with map plans): predict it again now and then while nothing is held, so that a bumped object is followed. This needs a plan head that is good on closed-loop frames too: predicting at every query was worse than keeping the plan (section 4 of the notebook of 13 Nov).
4. Make the plan head more precise (the tomato sauce): more cells or the wrist camera, and training on frames of closed-loop rollouts.
5. The ketchup's grasp and carry, which is now its own problem.